# Phase 6 — Interactive FitInsight notebook

**Primary Phase 6 artifact: an interactive Jupyter notebook.** Free `ipywidgets` controls filter already-validated Phase 1–5 records. The interface does not refit clusters, remine rules or recompute anomaly detectors. A Streamlit application remains paused until notebook conversion is reviewed.

The underlying sample contains 35 participants and 457 participant-days. These are convenience-sample activity records, not representative health data. Recorded coverage does not verify device wear; candidate flags are not diagnoses or confirmed errors.

In [1]:
from pathlib import Path
import csv,json,html,math
import ipywidgets as widgets
from IPython.display import display,HTML,SVG
def find_project_root(start=Path.cwd()):
    for candidate in (start,*start.parents):
        if (candidate/'reports/generated').is_dir() and (candidate/'data/processed').is_dir(): return candidate
    raise RuntimeError('Run from the FitInsight repository or a child directory')
ROOT=find_project_root()
def read_csv(path):
    with path.open(encoding='utf-8-sig',newline='') as stream: return list(csv.DictReader(stream))
def table(rows,columns,max_rows=12):
    rows=list(rows)[:max_rows]
    if not rows: return '<em>No rows match these filters.</em>'
    return '<table style="border-collapse:collapse"><tr>'+''.join(f'<th>{html.escape(c)}</th>' for c in columns)+'</tr>'+''.join('<tr>'+''.join(f'<td>{html.escape(str(row.get(c,"")))}</td>' for c in columns)+'</tr>' for row in rows)+'</table>'

def step_svg(rows,width=760,height=260):
    # A small dependency-free chart generated directly from the currently filtered rows.
    by_date={}
    for row in rows: by_date.setdefault(row['activity_date'],[]).append(float(row['total_steps']))
    points=sorted([(day,sum(values)/len(values)) for day,values in by_date.items()])
    if not points: return '<svg xmlns="http://www.w3.org/2000/svg" width="760" height="260"><text x="20" y="40">No rows to plot</text></svg>'
    lo=min(y for _,y in points); hi=max(y for _,y in points); span=max(hi-lo,1)
    plotted=[]
    for i,(_,value) in enumerate(points):
        x=45+i*(width-70)/max(len(points)-1,1); y=height-35-(value-lo)*(height-75)/span
        plotted.append(f'{x:.1f},{y:.1f}')
    poly=' '.join(plotted)
    return f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img"><title>Mean recorded steps by date</title><desc>{len(points)} filtered participant-day records; daily mean across the selected participant-days; descriptive only.</desc><rect width="100%" height="100%" fill="white"/><line x1="45" y1="20" x2="45" y2="{height-35}" stroke="#667"/><line x1="45" y1="{height-35}" x2="{width-20}" y2="{height-35}" stroke="#667"/><polyline fill="none" stroke="#277da1" stroke-width="2" points="{poly}"/><text x="48" y="18" font-size="12">max {hi:.0f} steps</text><text x="48" y="{height-12}" font-size="12">min {lo:.0f} steps • date means across filtered records</text></svg>'

activity=read_csv(ROOT/'data/processed/daily_activity_clean.csv')
clusters=read_csv(ROOT/'reports/generated/activity_clustering/tables/participant_clusters.csv')
rules=read_csv(ROOT/'reports/generated/associations/association_rules.csv')
candidates=read_csv(ROOT/'reports/generated/anomalies/candidate_registry.csv')
sleep=read_csv(ROOT/'reports/generated/eda/tables/sleep_episodes.csv')
time_accounting=read_csv(ROOT/'reports/generated/eda/tables/time_accounting_by_date.csv')
assert len(activity)==457 and len({r['id'] for r in activity})==35
print('Loaded stored evidence:',{'activity_days':len(activity),'clusters':len(clusters),'rules':len(rules),'anomaly_candidates':len(candidates),'sleep_episodes':len(sleep),'time_accounting_rows':len(time_accounting)})

Loaded stored evidence: {'activity_days': 457, 'clusters': 35, 'rules': 860, 'anomaly_candidates': 457, 'sleep_episodes': 556, 'time_accounting_rows': 23}


## Activity and recorded coverage

In [2]:
participant=widgets.Dropdown(options=['All participants']+sorted({r['id'] for r in activity}),description='Participant')
coverage=widgets.Dropdown(options=['All days','Complete only','Incomplete only'],description='Coverage')
out_activity=widgets.Output()
def render_activity(change=None):
    subset=[r for r in activity if (participant.value=='All participants' or r['id']==participant.value) and (coverage.value=='All days' or (r['is_complete_day']=='True')==(coverage.value=='Complete only'))]
    with out_activity:
        out_activity.clear_output(); steps=[int(float(r['total_steps'])) for r in subset]
        print(f'{len(subset)} participant-days; {len({r["id"] for r in subset})} participants; mean recorded steps={sum(steps)/len(steps):.1f}' if steps else 'No records')
        display(SVG(data=step_svg(subset))); display(HTML(table(subset,['id','activity_date','total_steps','total_recorded_minutes','is_complete_day'])))
participant.observe(render_activity,names='value');coverage.observe(render_activity,names='value');render_activity()
display(widgets.VBox([participant,coverage,out_activity]))

## Cluster assignments

In [3]:
cluster_cohort=widgets.Dropdown(options=['all_days','complete_days'],description='Cohort')
cluster_column={'all_days':'all_days_cluster','complete_days':'complete_days_cluster'}
all_cluster_labels=sorted({r[col] for r in clusters for col in cluster_column.values() if r.get(col)})
cluster_choice=widgets.Dropdown(options=['All clusters']+all_cluster_labels,description='Cluster')
out_cluster=widgets.Output()
def render_clusters(change=None):
    field=cluster_column[cluster_cohort.value]
    selected=[{'cohort':cluster_cohort.value,'id':r['id'],'cluster':r[field]} for r in clusters if cluster_choice.value=='All clusters' or r[field]==cluster_choice.value]
    with out_cluster:
        out_cluster.clear_output();print(f'{len(selected)} stored participant assignments; filtering does not refit the model.')
        display(HTML(table(selected,['cohort','id','cluster'],35)))
cluster_cohort.observe(render_clusters,names='value');cluster_choice.observe(render_clusters,names='value');render_clusters()
display(widgets.VBox([cluster_cohort,cluster_choice,out_cluster]))

## Association-rule filters

In [4]:
confidence=widgets.FloatSlider(value=.60,min=0,max=1,step=.05,description='Min confidence')
lift=widgets.FloatSlider(value=1.10,min=0,max=3,step=.05,description='Min lift')
out_rules=widgets.Output()
def render_rules(change=None):
    selected=sorted([r for r in rules if float(r['confidence'] or 0)>=confidence.value and float(r['lift'] or 0)>=lift.value],key=lambda r:(-float(r['lift'] or 0),-float(r['confidence'] or 0)))
    with out_rules:
        out_rules.clear_output();print(f'{len(selected)} stored rules meet the display thresholds; Apriori is not rerun.')
        display(HTML(table(selected,['antecedent_items','consequent_items','support','confidence','lift','threshold_eligible'])))
confidence.observe(render_rules,names='value');lift.observe(render_rules,names='value');render_rules()
display(widgets.VBox([confidence,lift,out_rules]))

## Anomaly-candidate review

In [5]:
candidate_methods=sorted({method for row in candidates for method in json.loads(row.get('candidate_methods') or '[]')})
method=widgets.Dropdown(options=['All methods']+candidate_methods,description='Screen')
out_candidates=widgets.Output()
def render_candidates(change=None):
    selected=[r for r in candidates if method.value=='All methods' or method.value in json.loads(r.get('candidate_methods') or '[]')]
    with out_candidates:
        out_candidates.clear_output();print(f'{len(selected)} candidate-days. A flag is a review prompt, not a confirmed error.')
        display(HTML(table(selected,['participant_id','date','complete_day','total_steps','total_recorded_minutes','candidate_methods'],15)))
method.observe(render_candidates,names='value');render_candidates();display(widgets.VBox([method,out_candidates]))

## Sleep and recorded-time accounting

In [6]:
date_options=sorted({r.get('activity_date','') for r in time_accounting if r.get('activity_date')})
sleep_date=widgets.Dropdown(options=['All dates']+date_options,description='Date')
out_sleep=widgets.Output()
def render_sleep(change=None):
    selected=[r for r in time_accounting if sleep_date.value=='All dates' or r.get('activity_date')==sleep_date.value]
    episodes=[r for r in sleep if sleep_date.value=='All dates' or r.get('calendar_date')==sleep_date.value or r.get('waking_date')==sleep_date.value]
    with out_sleep:
        out_sleep.clear_output();print(f'{len(selected)} time-accounting rows; {len(episodes)} recorded sleep episodes. Episode counts are not sleep duration.')
        display(HTML(table(selected,list(selected[0])[:7] if selected else [],12)));display(HTML(table(episodes,list(episodes[0])[:7] if episodes else [],10)))
sleep_date.observe(render_sleep,names='value');render_sleep();display(widgets.VBox([sleep_date,out_sleep]))

## Interactive overview

The panels below present controls and summaries for the stored evidence. Filtering changes only the view. To change analysis results, rerun the corresponding Phase 1–5 notebook and review the resulting validation evidence.

In [7]:
overview=widgets.VBox([widgets.HTML('<h3>FitInsight evidence overview</h3>'),widgets.HTML(f'<p>{len(activity)} participant-days from {len({r["id"] for r in activity})} participants; {len(clusters)} stored cluster profiles; {len(rules)} stored association rules; {len(candidates)} anomaly registry rows; {len(sleep)} sleep episodes.</p><p>Historical convenience sample. Recorded coverage is not verified wear; filters do not refit models. A candidate is not a confirmed error.</p>')])
tabs=widgets.Tab(children=[overview,widgets.VBox([widgets.HTML('<h3>Activity</h3>'),participant,coverage,out_activity]),widgets.VBox([widgets.HTML('<h3>Clusters</h3>'),cluster_cohort,cluster_choice,out_cluster]),widgets.VBox([widgets.HTML('<h3>Associations</h3>'),confidence,lift,out_rules]),widgets.VBox([widgets.HTML('<h3>Anomaly review</h3>'),method,out_candidates]),widgets.VBox([widgets.HTML('<h3>Sleep and time</h3>'),sleep_date,out_sleep])])
for i,title in enumerate(['Overview','Activity','Clusters','Associations','Anomalies','Sleep/time']):tabs.set_title(i,title)
display(tabs)
print('FITINSIGHT_PROOF | P6-INTERACTIVE-NOTEBOOK | panels=6 | dynamic_svg_chart=true | model_refit=false | external_analysis_packages=0')

FITINSIGHT_PROOF | P6-INTERACTIVE-NOTEBOOK | panels=6 | dynamic_svg_chart=true | model_refit=false | external_analysis_packages=0


## Limitations and next step

The dashboard is an evidence browser, not a clinical tool or a live data service. It deliberately avoids recomputing Phase 4–5 models in response to widgets. Phase 6 web-dashboard work stays paused until the notebook conversion is reviewed.